# Bronze - Google Trends 2025

## Objetivo

Realizar a ingestão dos dados públicos do Google Trends utilizados
para representar a evolução do interesse de busca no Brasil em 2025.

## Categorias analisadas

- Notícias
- Esportes
- Música
- Humor
- Jogos eletrônicos

## Configuração da coleta

- Localização: Brasil
- Período: 2025
- Pesquisa: Pesquisa Google na Web
- Fonte: Google Trends
- Arquivo: google_trends_2025_5_categorias.csv

Nesta camada o arquivo original será preservado no Volume.
Tratamentos de estrutura, datas e categorias serão realizados
posteriormente na camada Silver.

In [0]:
from pyspark.sql import functions as F

In [0]:
caminho_base_trends = (
    "/Volumes/workspace/mvp_bronze/raw_files/google_trends_2025"
)

display(
    dbutils.fs.ls(caminho_base_trends)
)

In [0]:
caminho_trends = (
    "/Volumes/workspace/mvp_bronze/raw_files/google_trends_2025/"
    "google_trends_2025_5_categorias.csv"
)

print(caminho_trends)

In [0]:
df_trends_raw_text = spark.read.text(caminho_trends)

display(
    df_trends_raw_text.limit(15)
)

In [0]:
df_trends = (
    spark.read
    .option("header", True)
    .option("sep", ",")
    .option("quote", '"')
    .option("escape", '"')
    .option("encoding", "UTF-8")
    .option("inferSchema", False)
    .csv(caminho_trends)
)

display(df_trends.limit(10))

In [0]:
print("Quantidade de colunas:", len(df_trends.columns))

for coluna in df_trends.columns:
    print(repr(coluna))

In [0]:
print("Quantidade de registros:", df_trends.count())

In [0]:
display(
    df_trends.limit(20)
)

In [0]:
from pyspark.sql import functions as F

df_trends_bronze = (
    df_trends
    .withColumn(
        "_data_ingestao",
        F.current_timestamp()
    )
    .withColumn(
        "_fonte",
        F.lit("Google Trends")
    )
    .withColumn(
        "_ano_referencia",
        F.lit(2025)
    )
    .withColumn(
        "_localizacao",
        F.lit("Brasil")
    )
    .withColumn(
        "_tipo_pesquisa",
        F.lit("Pesquisa Google na Web")
    )
    .withColumn(
        "_arquivo_origem",
        F.lit("google_trends_2025_5_categorias.csv")
    )
)

In [0]:
import re
import unicodedata

def normalizar_nome_coluna(nome):
    # Remove acentos
    nome_normalizado = (
        unicodedata
        .normalize("NFKD", nome)
        .encode("ASCII", "ignore")
        .decode("ASCII")
    )

    # Troca espaços e caracteres especiais por _
    nome_normalizado = re.sub(
        r"[^A-Za-z0-9_]",
        "_",
        nome_normalizado
    )

    # Remove underscores duplicados
    nome_normalizado = re.sub(
        r"_+",
        "_",
        nome_normalizado
    )

    # Padroniza em minúsculas
    return nome_normalizado.lower()


for coluna in df_trends_bronze.columns:
    novo_nome = normalizar_nome_coluna(coluna)

    if coluna != novo_nome:
        print(f"{coluna}  -->  {novo_nome}")

    df_trends_bronze = (
        df_trends_bronze
        .withColumnRenamed(coluna, novo_nome)
    )

In [0]:
(
    df_trends_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.mvp_bronze.google_trends_2025"
    )
)

In [0]:
df_trends_check = spark.table(
    "workspace.mvp_bronze.google_trends_2025"
)

print(
    "Linhas:",
    df_trends_check.count()
)

print(
    "Colunas:",
    len(df_trends_check.columns)
)

display(
    df_trends_check.limit(10)
)

In [0]:
%sql

DROP TABLE IF EXISTS workspace.mvp_bronze.youtube_categorias_br;

DROP TABLE IF EXISTS workspace.mvp_bronze.youtube_trending_br;

In [0]:
%sql

SHOW TABLES IN workspace.mvp_bronze;

### Padronização técnica dos nomes das colunas

Durante a criação da tabela Delta foi identificado que os nomes originais
das colunas continham caracteres incompatíveis com o padrão de nomes
aceito pela tabela, como espaços e acentos.

O arquivo bruto original foi mantido sem alteração no Volume.

Para possibilitar seu registro como tabela Delta, somente os nomes técnicos
dos campos foram normalizados.

Exemplos:

- `Notícia` → `noticia`
- `Música` → `musica`
- `Jogo eletrônico` → `jogo_eletronico`

Os valores originais dos dados não foram modificados nesta etapa.